In [1]:
import numpy as np

train = np.loadtxt("Datasets/pendigits.tra", delimiter=",")
test = np.loadtxt("Datasets/pendigits.tes", delimiter=",")
Xtr, ytr = train[:, :16] / 100, train[:, 16].astype(int)
Xte, yte = test[:, :16] / 100, test[:, 16].astype(int)
n = len(Xtr)

print("Training samples:", n, " Testing samples:", len(Xte), " Features:", Xtr.shape[1])

np.random.seed(1)
W1 = np.random.randn(16, 64) * np.sqrt(2 / 16)
b1 = np.zeros(64)
W2 = np.random.randn(64, 10) * np.sqrt(2 / 64)
b2 = np.zeros(10)
lr = 0.1
epochs = 45
batch = 128


def relu(z):
    return np.maximum(0, z)


def softmax(z):
    e = np.exp(z - z.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)


def predict(X):
    h = relu(np.dot(X, W1) + b1)
    return softmax(np.dot(h, W2) + b2)


for epoch in range(epochs):
    p = np.random.permutation(n)
    for i in range(0, n, batch):
        idx = p[i:i + batch]
        x = Xtr[idx]
        m = len(idx)
        h = relu(np.dot(x, W1) + b1)
        out = softmax(np.dot(h, W2) + b2)
        out[np.arange(m), ytr[idx]] -= 1
        d2 = out / m
        d1 = np.dot(d2, W2.T) * (h > 0)
        W2 = W2 - lr * np.dot(h.T, d2)
        b2 = b2 - lr * d2.sum(axis=0)
        W1 = W1 - lr * np.dot(x.T, d1)
        b1 = b1 - lr * d1.sum(axis=0)
    if epoch % 5 == 0:
        prob = predict(Xtr)
        loss = -np.mean(np.log(prob[np.arange(n), ytr] + 1e-9))
        print("Epoch", epoch, " Loss =", round(loss, 4))

pred_tr = predict(Xtr).argmax(axis=1)
pred_te = predict(Xte).argmax(axis=1)
print("Training Accuracy:", round((pred_tr == ytr).mean(), 4))
print("Test Accuracy:", round((pred_te == yte).mean(), 4))


Training samples: 7494  Testing samples: 3498  Features: 16
Epoch 0  Loss = 1.1292
Epoch 5  Loss = 0.4088
Epoch 10  Loss = 0.292
Epoch 15  Loss = 0.2406
Epoch 20  Loss = 0.2059
Epoch 25  Loss = 0.1818
Epoch 30  Loss = 0.1608
Epoch 35  Loss = 0.1452
Epoch 40  Loss = 0.1314
Training Accuracy: 0.97
Test Accuracy: 0.9314
